In [149]:
import pandas as pd
import numpy as np

In [186]:
# START_DATE = '2025-10-20'
# END_DATE = '2026-04-20'
START_DATE = '2025-04-20'
END_DATE = '2025-07-20'
GET_ALL_DATES_FROM_MB51 = False

PRODUCTION_PLANT = '2101'

# 90% -> k = 1.28
# 95% -> k = 1.65 (najczęstszy wybór)
# 99% -> k = 2.33
K_PARAMETER = 2.33


In [187]:
# Mapowanie dla mb51_df (Snake Case)
mb51_rename = {
    'Zakład': 'plant',
    'Materiał': 'material',
    'Opis materiału': 'material_description',
    'Data księgowania': 'posting_date',
    'Ilość': 'quantity',
    'Podst. jedn. miary': 'base_uom',
    'Rodzaj ruchu': 'movement_type'
}

# Mapowanie dla zsbe_df (Snake Case + Cena Jednostkowa)
zsbe_rename = {
    'Materiał': 'material',
    'Opis materiału': 'material_description',
    'Rodzaj materiału': 'material_type',
    'Zakład': 'plant',
    'Planow. czas dostawy': 'planned_delivery_time',
    'Całk. czas uzupełn.': 'total_replenishment_time',
    'Unnamed: 6': 'unit_price',  # Poprawka zgodnie z Twoją informacją
    'Waluta': 'currency',
    'Jednostka ceny': 'price_unit',
    'dowolne użycie': 'unrestricted_use',
    'Podst. jedn. miary': 'base_uom',
    'pokrycie/M': 'coverage_month',
    'przec.ilość/MM': 'avg_qty_month',
    'zapas bezpieczeństwa': 'safety_stock',
    'Kontroler MRP': 'mrp_controller'
}

mb51_path = r"P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\15_SAFETY_STOCKS_CALCULATIONS\L1K_Consumption.XLSX"
zsbe_path = r"P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\15_SAFETY_STOCKS_CALCULATIONS\L1K_items_and_parameters.XLSX"

In [188]:
mb51_df = pd.read_excel(mb51_path, dtype={'Materiał': str, 'Zakład': str})
zsbe_df = pd.read_excel(zsbe_path, dtype={'Materiał': str, 'Zakład': str})

In [189]:
# Aplikacja zmian
mb51_df.rename(columns=mb51_rename, inplace=True)
zsbe_df.rename(columns=zsbe_rename, inplace=True)

In [190]:
# Przygotowujemy unikalną listę materiałów i ich typów
unique_materials = zsbe_df[['material', 'material_type']].drop_duplicates(subset=['material'])

In [191]:
mb51_df = pd.merge(left=mb51_df, right=unique_materials, on='material', how='left')

In [192]:
# Zachowujemy wiersze, które NIE spełniają obu warunków jednocześnie
mb51_df = mb51_df[~((mb51_df['movement_type'] == 261) & (mb51_df['material_type'] == 'FERT'))]

In [193]:
# Konwersja na datę (jeśli jeszcze nie zrobiona)
mb51_df['posting_date'] = pd.to_datetime(mb51_df['posting_date'])

# Opcjonalnie: zmiana znaku na dodatni, jeśli quantity to wydania (ujemne)
mb51_df['quantity'] = mb51_df['quantity'].abs()

In [194]:
# 1. Pobierz unikalne pary Materiał-Zakład (Twoje 3581 wierszy)
unique_pairs = zsbe_df[['material', 'plant']].drop_duplicates()

# 2. Pobierz unikalne daty (Twoje 187 dat)
if GET_ALL_DATES_FROM_MB51:
    all_dates = mb51_df['posting_date'].unique()
else:
    all_dates = pd.date_range(start=START_DATE, end=END_DATE, freq='B')

# 3. Stwórz stelaż bezpośrednio z par i dat
# Tworzymy listę krotek (material, plant, date)
from itertools import product

structure = [
    (m, p, d) for (m, p), d in product(unique_pairs.values, all_dates)
]

# 4. Zamień na DataFrame
full_frame = pd.DataFrame(structure, columns=['material', 'plant', 'posting_date'])

In [195]:
# Agregacja MB51, żeby mieć jedną sumę na dzień/materiał/zakład
daily_actual = mb51_df.groupby(['material', 'plant', 'posting_date'])['quantity'].sum().reset_index()

# Połączenie
final_df = pd.merge(
    full_frame,
    daily_actual,
    on=['material', 'plant', 'posting_date'],
    how='left'
).fillna(0)

In [196]:
# 1. Grupowanie po materiale i zakładzie w celu obliczenia statystyk
stats_df = final_df.groupby(['material', 'plant'])['quantity'].agg(['mean', 'std']).reset_index()

# 2. Zmiana nazw kolumn na bardziej opisowe (Snake Case)
stats_df.rename(columns={
    'mean': 'daily_avg_consumption',
    'std': 'daily_std_dev'
}, inplace=True)

# 3. Obsługa wartości pustych (jeśli std nie mogło zostać wyliczone, np. brak zmienności)
stats_df['daily_std_dev'] = stats_df['daily_std_dev'].fillna(0)

# 4. Opcjonalnie: Zaokrąglenie wyników do 2 miejsc po przecinku dla lepszej czytelności
stats_df['daily_avg_consumption'] = stats_df['daily_avg_consumption'].round(4)
stats_df['daily_std_dev'] = stats_df['daily_std_dev'].round(4)

# Sprawdzenie wyniku
print(f"Liczba wynikowych wierszy: {len(stats_df)}") # Powinno być 3581
stats_df.sort_values(by='daily_avg_consumption', ascending=False, inplace=True)
stats_df.reset_index(drop=True, inplace=True)
# stats_df

Liczba wynikowych wierszy: 3581


In [197]:
lead_times = zsbe_df[['material', 'plant', 'planned_delivery_time', 'total_replenishment_time']].drop_duplicates()

In [198]:
# Tworzymy kolumnę 'lead_time' na podstawie warunku
lead_times['lead_time'] = np.where(
    lead_times['plant'] == PRODUCTION_PLANT,                  # Warunek: czy zakład to 2101
    lead_times['total_replenishment_time'],       # Jeśli prawda: bierz czas uzupełnienia
    lead_times['planned_delivery_time']           # Jeśli fałsz: bierz czas dostawy
)

# Opcjonalnie: Jeśli masz w danych wartości NaN (puste), warto je wypełnić zerem,
# aby pierwiastek w obliczeniach SS nie wyrzucił błędu
lead_times['lead_time'] = lead_times['lead_time'].fillna(0)

# Teraz możesz połączyć to ze swoją tabelą statystyk
stats_df = pd.merge(stats_df, lead_times[['material', 'plant', 'lead_time']], on=['material', 'plant'], how='left')

In [199]:
stats_df['safety_stock'] = (
    K_PARAMETER * stats_df['daily_std_dev'] * np.sqrt(stats_df['lead_time'])
)

# 4. Zaokrąglamy w górę (bo nie można mieć 0.5 sztuki na zapasie)
stats_df['safety_stock'] = np.ceil(stats_df['safety_stock']).astype(int)

stats_df[['material', 'plant', 'daily_avg_consumption', 'safety_stock']].head()

,material,plant,daily_avg_consumption,safety_stock
0,808965,3701,16.3538,205
1,808965,2101,11.4615,48
2,839134,2101,9.6308,37
3,808966,2101,8.0154,30
4,839135,2101,5.6308,24


In [201]:
stats_df.head(20)

,material,plant,daily_avg_consumption,daily_std_dev,lead_time,safety_stock
0,808965,3701,16.3538,27.8127,10,205
1,808965,2101,11.4615,14.3887,2,48
2,839134,2101,9.6308,11.1040,2,37
3,808966,2101,8.0154,9.0804,2,30
4,839135,2101,5.6308,7.2361,2,24
5,808966,3701,5.4769,9.0642,3,37
6,990510,2101,4.8308,6.2063,1,15
7,808964,3701,2.2308,2.8819,4,14
8,741480,0301,1.6923,2.3778,4,12
9,990495,0301,1.6769,1.8550,4,9
